In [4]:
## Install missingno for missing value visualization
!pip install missingno -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
import warnings

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (12, 5)
sns.set_style("whitegrid")

print("All libraries imported successfully!")

All libraries imported successfully!


## 📊 Data Visualization & Insights


In [7]:
# Load all required CSV files
df_players       = pd.read_csv('/kaggle/input/datasets/davidcariboo/player-scores/players.csv')
df_valuations    = pd.read_csv('/kaggle/input/datasets/davidcariboo/player-scores/player_valuations.csv')
df_appearances   = pd.read_csv('/kaggle/input/datasets/davidcariboo/player-scores/appearances.csv')
df_competitions  = pd.read_csv('/kaggle/input/datasets/davidcariboo/player-scores/competitions.csv')
df_clubs         = pd.read_csv('/kaggle/input/datasets/davidcariboo/player-scores/clubs.csv')
df_transfers     = pd.read_csv('/kaggle/input/datasets/davidcariboo/player-scores/transfers.csv')

print("All datasets loaded successfully!")
print(f"players:        {df_players.shape}")
print(f"valuations:     {df_valuations.shape}")
print(f"appearances:    {df_appearances.shape}")
print(f"competitions:   {df_competitions.shape}")
print(f"clubs:          {df_clubs.shape}")
print(f"transfers:      {df_transfers.shape}")

All datasets loaded successfully!
players:        (34376, 23)
valuations:     (525510, 6)
appearances:    (1747138, 13)
competitions:   (44, 11)
clubs:          (451, 17)
transfers:      (100266, 10)


In [ ]:
#Print Top Rows of Each Dataset
print("=" * 60)
print("PLAYERS.CSV - Top 5 Rows")
print("=" * 60)
display(df_players.head())

print("=" * 60)
print("PLAYER_VALUATIONS.CSV - Top 5 Rows")
print("=" * 60)
display(df_valuations.head())

print("=" * 60)
print("APPEARANCES.CSV - Top 5 Rows")
print("=" * 60)
display(df_appearances.head())

print("=" * 60)
print("COMPETITIONS.CSV - Top 5 Rows")
print("=" * 60)
display(df_competitions.head())

print("=" * 60)
print("CLUBS.CSV - Top 5 Rows")
print("=" * 60)
display(df_clubs.head())

print("=" * 60)
print("TRANSFERS.CSV - Top 5 Rows")
print("=" * 60)
display(df_transfers.head())

In [ ]:
#Statistical Description (.describe())

print("=" * 60)
print("PLAYERS - Statistical Description")
print("=" * 60)
display(df_players.describe(include='all'))

print("=" * 60)
print("PLAYER VALUATIONS - Statistical Description")
print("=" * 60)
display(df_valuations.describe(include='all'))

print("=" * 60)
print("APPEARANCES - Statistical Description")
print("=" * 60)
display(df_appearances.describe(include='all'))

print("=" * 60)
print("CLUBS - Statistical Description")
print("=" * 60)
display(df_clubs.describe(include='all'))

print("=" * 60)
print("TRANSFERS - Statistical Description")
print("=" * 60)
display(df_transfers.describe(include='all'))

In [ ]:
#Checking for missing Values

datasets = {
    'players': df_players,
    'player_valuations': df_valuations,
    'appearances': df_appearances,
    'competitions': df_competitions,
    'clubs': df_clubs,
    'transfers': df_transfers
}

for name, df in datasets.items():
    missing = df.isnull().sum()
    missing_pct = (df.isnull().sum() / len(df)) * 100
    missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
    missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)
    
    if not missing_df.empty:
        print(f"\n{'='*60}")
        print(f"Missing Values in: {name.upper()}")
        print(f"{'='*60}")
        display(missing_df)
    else:
        print(f"\n{name.upper()} — No missing values found!")

In [ ]:
#Checking DataTypes 

for name, df in datasets.items():
    print(f"\n{'='*60}")
    print(f"Data Types: {name.upper()}")
    print(f"{'='*60}")
    print(df.dtypes)

## Data Visualization



In [ ]:
#Missing Value Heatmap and Barchart 

plt.figure(figsize=(14, 6))
msno.matrix(df_players)
plt.title("Missing Values in players.csv", fontsize=14)
plt.tight_layout()
plt.show()

missing_pct = df_players.isnull().mean() * 100
missing_pct = missing_pct[missing_pct > 0].sort_values(ascending=False)

plt.figure(figsize=(12, 5))
sns.barplot(x=missing_pct.values, y=missing_pct.index, palette='Reds_r')
plt.xlabel("Missing Percentage (%)")
plt.title("Missing Value Percentage per Column - players.csv")
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of Market Value
df_mv = df_players.dropna(subset=['market_value_in_eur'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw distribution
axes[0].hist(df_mv['market_value_in_eur'], bins=100, color='steelblue', edgecolor='white')
axes[0].set_title("Market Value Distribution (Raw)")
axes[0].set_xlabel("Market Value (EUR)")
axes[0].set_ylabel("Number of Players")

# Log distribution (better to visualize skewed data)
axes[1].hist(np.log1p(df_mv['market_value_in_eur']), bins=100, color='darkorange', edgecolor='white')
axes[1].set_title("Market Value Distribution (Log Scale)")
axes[1].set_xlabel("Log(Market Value)")
axes[1].set_ylabel("Number of Players")

plt.tight_layout()
plt.show()

# Note: Market value is heavily right-skewed. Most players have low values while a small number of elite players have extremely high values. Log transformation makes the distribution more normal — this will be useful for the ML model.

In [ ]:
#Market Value by Position

df_pos = df_players.dropna(subset=['market_value_in_eur', 'position'])

plt.figure(figsize=(12, 5))
sns.boxplot(x='position', y='market_value_in_eur', data=df_pos, palette='Set2')
plt.title("Market Value by Position")
plt.xlabel("Position")
plt.ylabel("Market Value (EUR)")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

#Note: Attackers and midfielders generally command higher market values compared to goalkeepers and defenders.

In [ ]:
#Age vs Market Value
# Calculate age first
df_players['date_of_birth'] = pd.to_datetime(df_players['date_of_birth'], errors='coerce')
df_players['age'] = (pd.Timestamp.now() - df_players['date_of_birth']).dt.days // 365

df_age = df_players.dropna(subset=['age', 'market_value_in_eur'])
df_age = df_age[(df_age['age'] >= 15) & (df_age['age'] <= 45)]

plt.figure(figsize=(12, 5))
sns.scatterplot(x='age', y='market_value_in_eur', data=df_age, alpha=0.4, color='purple')
plt.title("Age vs Market Value")
plt.xlabel("Age (Years)")
plt.ylabel("Market Value (EUR)")
plt.tight_layout()
plt.show()

#Note: Market value peaks around ages 23–28 and declines significantly after 30. This bell-curve pattern is expected in football.

In [ ]:
#Age Distribution of Players

plt.figure(figsize=(12, 5))
sns.histplot(df_players['age'].dropna(), bins=40, kde=True, color='teal')
plt.title("Age Distribution of All Players")
plt.xlabel("Age")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
#Goals & Assists vs Market Value

# Aggregate appearances per player
agg = df_appearances.groupby('player_id').agg(
    total_goals=('goals', 'sum'),
    total_assists=('assists', 'sum'),
    total_red_cards=('red_cards', 'sum'),
    total_yellow_cards=('yellow_cards', 'sum'),
    total_minutes=('minutes_played', 'sum')
).reset_index()

# Merge with players
df_merged_temp = df_players.dropna(subset=['market_value_in_eur']).merge(agg, on='player_id', how='left')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(df_merged_temp['total_goals'], df_merged_temp['market_value_in_eur'], alpha=0.3, color='green')
axes[0].set_title("Total Goals vs Market Value")
axes[0].set_xlabel("Total Goals")
axes[0].set_ylabel("Market Value (EUR)")

axes[1].scatter(df_merged_temp['total_assists'], df_merged_temp['market_value_in_eur'], alpha=0.3, color='blue')
axes[1].set_title("Total Assists vs Market Value")
axes[1].set_xlabel("Total Assists")
axes[1].set_ylabel("Market Value (EUR)")

plt.tight_layout()
plt.show()

#Note: There is a positive correlation — players with more goals and assists tend to have higher market values, especially strikers and attacking midfielders.

In [ ]:
# ============================================================
# Red Cards vs Market Value (Grouped Boxplot)
# We group players by how many red cards they have received
# and compare their market values. This shows whether
# disciplinary issues negatively impact a player's value.
# ============================================================

plt.figure(figsize=(10, 5))
sns.boxplot(
    x=pd.cut(df_merged_temp['total_red_cards'],
             bins=[-1, 0, 2, 5, 100],
             labels=['0', '1-2', '3-5', '5+']),
    y='market_value_in_eur',
    data=df_merged_temp,
    palette='Reds'
)
plt.title("Red Cards vs Market Value")
plt.xlabel("Total Red Cards Group")
plt.ylabel("Market Value (EUR)")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Top 10 Clubs by Total Squad Market Value
# A horizontal bar chart ranking clubs by combined squad value.
# This helps identify which clubs have the highest quality
# squads and answers our research question about top clubs.
# ============================================================

df_clubs_clean_viz = df_clubs.dropna(subset=['total_market_value'])
top_clubs = df_clubs_clean_viz.nlargest(10, 'total_market_value')

plt.figure(figsize=(12, 5))
sns.barplot(x='total_market_value', y='name', data=top_clubs, palette='Blues_r')
plt.title("Top 10 Clubs by Total Market Value")
plt.xlabel("Total Market Value (EUR)")
plt.ylabel("Club")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
#  Career Market Value Trend for a Single Player
# We pick the player with the most valuation records and plot
# their value over time. This directly answers our research
# question: "How does market value change throughout a career?"
# ============================================================

df_valuations['date'] = pd.to_datetime(df_valuations['date'], errors='coerce')

top_player_id = df_valuations['player_id'].value_counts().idxmax()
sample = df_valuations[df_valuations['player_id'] == top_player_id].sort_values('date')

plt.figure(figsize=(12, 5))
plt.plot(sample['date'], sample['market_value_in_eur'], marker='o', color='crimson')
plt.title(f"Career Market Value Trend - Player ID: {top_player_id}")
plt.xlabel("Date")
plt.ylabel("Market Value (EUR)")
plt.tight_layout()
plt.show()

#Note: This plot shows how a single player's market value rises and falls throughout their career — useful for time-series analysis in later phases.

In [ ]:
# ============================================================
# Number of Competitions by Type
# This bar chart shows how many domestic, international, and
# other competition types exist in the dataset. Helps us
# understand the competition level diversity in our data.
# ============================================================

df_comp_clean_viz = df_competitions.dropna(subset=['type'])

plt.figure(figsize=(10, 5))
df_comp_clean_viz['type'].value_counts().plot(kind='bar', color='coral', edgecolor='black')
plt.title("Number of Competitions by Type")
plt.xlabel("Competition Type")
plt.ylabel("Count")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Player Preferred Foot Distribution (Pie Chart)
# Shows the ratio of right-footed, left-footed and both-footed
# players. Left-footed players are rarer and often valued
# higher — this feature may have ML signal.
# ============================================================

plt.figure(figsize=(7, 5))
df_players['foot'].value_counts().plot(
    kind='pie',
    autopct='%1.1f%%',
    colors=['steelblue', 'salmon', 'lightgreen'],
    startangle=90
)
plt.title("Player Preferred Foot Distribution")
plt.ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Correlation Heatmap of All Key Numeric Features
# Shows how strongly each feature is correlated with others
# and with market value. Strong correlations (close to 1 or -1)
# indicate important features for the ML model.
# ============================================================

numeric_cols = ['market_value_in_eur', 'age', 'height_in_cm',
                'total_goals', 'total_assists', 'total_red_cards',
                'total_yellow_cards', 'total_minutes']

corr_df = df_merged_temp[numeric_cols].dropna()

plt.figure(figsize=(12, 8))
sns.heatmap(corr_df.corr(), annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title("Correlation Heatmap of Key Features")
plt.tight_layout()
plt.show()

#Note: total_goals, total_assists, and total_minutes show positive correlation with market value. Age shows a moderate relationship.

In [ ]:
# ============================================================
# Drop Irrelevant Columns from players.csv
# REASON: image_url, url, player_code are web links and system
# identifiers with zero predictive value. first_name and
# last_name are redundant since the 'name' column exists.
# city_of_birth has too many unique values (high cardinality)
# and very little signal for market value prediction.
# ============================================================

cols_to_drop = ['first_name', 'last_name', 'player_code',
                'image_url', 'url', 'city_of_birth']

df_players.drop(columns=[c for c in cols_to_drop if c in df_players.columns], inplace=True)

print("Remaining columns in players.csv:")
print(df_players.columns.tolist())

In [ ]:
# ============================================================
# Remove Rows with Missing Market Value
# REASON: market_value_in_eur is our target variable (the value
# we want to predict). Any row without this value is completely
# useless for supervised ML training and must be removed.
# ============================================================

before = len(df_players)
df_players.dropna(subset=['market_value_in_eur'], inplace=True)
after = len(df_players)

print(f"Rows before: {before}")
print(f"Rows after:  {after}")
print(f"Rows dropped: {before - after}")

In [ ]:
# ============================================================
# Handle Missing Values in players.csv
# REASON: Columns with >50% missing cannot provide reliable
# information to a model and are dropped. height_in_cm is
# filled with median (not mean) to avoid outlier influence.
# foot and position are filled with mode (most common value)
# since they are categorical columns.
# ============================================================

# Drop columns where more than 50% data is missing
threshold = 0.5
cols_before = df_players.shape[1]
df_players = df_players.dropna(thresh=int(len(df_players) * threshold), axis=1)
cols_after = df_players.shape[1]
print(f"Columns dropped due to >50% missing: {cols_before - cols_after}")

# Fill height with median
if 'height_in_cm' in df_players.columns:
    median_height = df_players['height_in_cm'].median()
    df_players['height_in_cm'].fillna(median_height, inplace=True)
    print(f"Filled height_in_cm with median: {median_height}")

# Fill foot with mode
if 'foot' in df_players.columns:
    df_players['foot'].fillna(df_players['foot'].mode()[0], inplace=True)
    print(f"Filled foot with mode: {df_players['foot'].mode()[0]}")

# Fill position with mode
if 'position' in df_players.columns:
    df_players['position'].fillna(df_players['position'].mode()[0], inplace=True)

print("\nRemaining missing values in players.csv:")
print(df_players.isnull().sum()[df_players.isnull().sum() > 0])

In [ ]:
# ============================================================
# Convert date_of_birth Column to Numeric Age
# REASON: ML models cannot process raw date strings. We convert
# date_of_birth into a numeric 'age' feature (in years) which
# the model can directly use. The original date column is then
# dropped since age captures all the needed information.
# ============================================================

df_players['date_of_birth'] = pd.to_datetime(df_players['date_of_birth'], errors='coerce')
df_players['age'] = (pd.Timestamp.now() - df_players['date_of_birth']).dt.days // 365
df_players.drop(columns=['date_of_birth'], inplace=True)

print("Age column created successfully. Sample values:")
print(df_players['age'].describe())

In [ ]:
# ============================================================
# Label Encode Categorical Columns
# REASON: ML algorithms require numeric inputs. Columns like
# position ('Attack', 'Defence'), foot ('Left', 'Right'), and
# country_of_citizenship contain text values. We use Label
# Encoding to convert them to numeric codes the model can use.
# ============================================================

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
cat_cols = ['position', 'sub_position', 'foot', 'country_of_citizenship']

for col in cat_cols:
    if col in df_players.columns:
        df_players[col] = df_players[col].astype(str)
        df_players[col + '_encoded'] = le.fit_transform(df_players[col])
        print(f"Encoded: {col}")

print("\nEncoding complete!")

In [ ]:
# ============================================================
# Aggregate Player Performance Stats from appearances.csv
# REASON: appearances.csv has one row per match per player which
# means thousands of rows per player. For ML, we need one row
# per player. We aggregate by summing goals, assists, cards and
# minutes so each player has a single performance summary row.
# ============================================================

agg_appearances = df_appearances.groupby('player_id').agg(
    total_goals=('goals', 'sum'),
    total_assists=('assists', 'sum'),
    total_red_cards=('red_cards', 'sum'),
    total_yellow_cards=('yellow_cards', 'sum'),
    total_minutes_played=('minutes_played', 'sum'),
    total_games=('appearance_id', 'count')
).reset_index()

print("Aggregated appearances shape:", agg_appearances.shape)
display(agg_appearances.head())

In [ ]:
# ============================================================
# Clean clubs.csv - Keep Only Relevant Columns
# REASON: We only keep columns that describe club quality and
# prestige which can influence a player's market value.
# Columns like stadium_name, coach_name, url are irrelevant
# to predicting player market value and are dropped.
# ============================================================

df_clubs_clean = df_clubs[['club_id', 'name', 'domestic_competition_id',
                             'total_market_value', 'squad_size', 'average_age',
                             'foreigners_percentage', 'national_team_players',
                             'net_transfer_record', 'last_season']].copy()

df_clubs_clean.rename(columns={
    'name': 'club_name',
    'last_season': 'club_last_season'
}, inplace=True)

print("Clubs cleaned shape:", df_clubs_clean.shape)
display(df_clubs_clean.head())

In [ ]:
# ============================================================
# Clean competitions.csv - Keep Only Relevant Columns
# REASON: We keep league-level features that indicate competition
# prestige such as confederation, country, and whether it is a
# major league. Columns like url and competition_code are system
# identifiers with no value for market value prediction.
# ============================================================

df_comp_clean = df_competitions[['competition_id', 'name', 'country_name',
                                   'confederation', 'is_major_national_league', 'type']].copy()

df_comp_clean.rename(columns={'name': 'competition_name'}, inplace=True)

print("Competitions cleaned shape:", df_comp_clean.shape)
display(df_comp_clean.head())

In [ ]:
# ============================================================
# Merge All Cleaned Datasets into One Master DataFrame
# REASON: ML models need all features in a single table. We
# join players with performance stats (appearances), club quality
# (clubs), and league prestige (competitions) using player_id,
# club_id and competition_id as foreign keys. Performance NaNs
# are filled with 0 for players who have no appearance records.
# ============================================================

df_master = df_players.copy()

# Merge aggregated appearances
df_master = df_master.merge(agg_appearances, on='player_id', how='left')
print(f"After merging appearances: {df_master.shape}")

# Merge clubs info
df_master = df_master.merge(df_clubs_clean, left_on='current_club_id',
                             right_on='club_id', how='left')
print(f"After merging clubs: {df_master.shape}")

# Merge competition info
df_master = df_master.merge(df_comp_clean,
                             left_on='current_club_domestic_competition_id',
                             right_on='competition_id', how='left')
print(f"After merging competitions: {df_master.shape}")

# Fill performance NaN values with 0
perf_cols = ['total_goals', 'total_assists', 'total_red_cards',
             'total_yellow_cards', 'total_minutes_played', 'total_games']
df_master[perf_cols] = df_master[perf_cols].fillna(0)

print("\nFinal Master DataFrame shape:", df_master.shape)
display(df_master.head())

In [ ]:
# ============================================================
# Final Summary Check of the Master DataFrame
# We verify the shape, data types, remaining missing values
# and target variable stats before saving to CSV.
# ============================================================

print("=" * 60)
print("FINAL MASTER DATAFRAME SUMMARY")
print("=" * 60)
print(f"Shape: {df_master.shape}")
print(f"\nData Types:\n{df_master.dtypes}")
print(f"\nMissing Values:\n{df_master.isnull().sum()[df_master.isnull().sum() > 0]}")
print(f"\nTarget Variable Stats:")
print(df_master['market_value_in_eur'].describe())

In [ ]:
# ============================================================
# Save the Final Preprocessed Dataset as CSV
# REASON: Saving the cleaned and merged dataset means we do not
# have to repeat all loading, cleaning and merging steps in
# Phase 3. We simply load this one file directly which saves
# significant time and computation in future phases.
# ============================================================

df_master.to_csv('preprocessed_player_data.csv', index=False)
print("Preprocessed dataset saved as 'preprocessed_player_data.csv'")
print(f"Final shape: {df_master.shape}")
print(f"Columns saved: {df_master.columns.tolist()}")

In [ ]:
# ============================================================
# Verify the Saved CSV File Loads Correctly
# We reload the saved file and check its shape and top rows
# to confirm everything was saved properly before finishing.
# ============================================================

df_check = pd.read_csv('preprocessed_player_data.csv')
print("File loaded successfully!")
print(f"Shape: {df_check.shape}")
display(df_check.head())